In [ ]:
import pandas as pd
import numpy as np
import random
import os

def get_expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

def update_elo(rating_a, rating_b, result, k=32):
    expected_a = get_expected_score(rating_a, rating_b)
    return rating_a + k * (result - expected_a), rating_b + k * ((1 - result) - (1 - expected_a))

# def run_interactive_ranking(elo_df, titles, star_rating):
#     bucket = elo_df[elo_df['original_rating'] == star_rating].index.tolist()
#     if len(bucket) < 2: return elo_df

#     print(f"--- Ranking {star_rating}-Star Books ({len(bucket)} books) ---")
#     while True:
#         try:
#             # Weighted random choice favoring fewer matches
#             weights = 1 / (elo_df.loc[bucket, 'matches_played'] + 1)
#             idx_a, idx_b = random.choices(bucket, weights=weights, k=2)
#             if idx_a == idx_b: continue
            
#             choice = input(f"[1]{titles.get(elo_df.at[idx_a, 'book_id'])} vs [2]{titles.get(elo_df.at[idx_b, 'book_id'])}").strip().lower()

#             if choice == 'q': break
#             if choice not in ['1', '2']: continue

#             # Update State
#             elo_df.loc[[idx_a, idx_b], 'matches_played'] += 1
#             ra, rb = elo_df.at[idx_a, 'elo_score'], elo_df.at[idx_b, 'elo_score']
#             new_ra, new_rb = update_elo(ra, rb, 1 if choice == '1' else 0)
#             elo_df.at[idx_a, 'elo_score'], elo_df.at[idx_b, 'elo_score'] = new_ra, new_rb
            
#         except KeyboardInterrupt:
#             break
#     return elo_df

def run_interactive_ranking(elo_df, titles, star_rating):
    bucket_indices = elo_df[elo_df['original_rating'] == star_rating].index.tolist()
    if len(bucket_indices) < 2: 
        return elo_df

    matches_stats = elo_df.loc[bucket_indices, 'matches_played']
    previous_min_matches = matches_stats.min()
    unranked_count = (matches_stats == 0).sum()
    
    print(f"--- Ranking {star_rating}-Star Books ({unranked_count} books with 0 matches) ---")
    while True:
        try:
            unranked_indices = [idx for idx in bucket_indices if elo_df.at[idx, 'matches_played'] == 0]

            if len(unranked_indices) >= 2:
                index_a, index_b = random.sample(unranked_indices, 2)
            elif len(unranked_indices) == 1:
                index_a = unranked_indices[0]
                remaining_indices = [idx for idx in bucket_indices if idx != index_a]
                match_weights = 1 / (elo_df.loc[remaining_indices, 'matches_played'] + 1)
                index_b = random.choices(remaining_indices, weights=match_weights, k=1)[0]
            else:
                match_weights = 1 / (elo_df.loc[bucket_indices, 'matches_played'] + 1)
                index_a, index_b = random.choices(bucket_indices, weights=match_weights, k=2)
                if index_a == index_b: 
                    continue
            
            title_a = titles.get(elo_df.at[index_a, 'book_id'])
            title_b = titles.get(elo_df.at[index_b, 'book_id'])

            user_choice = input(f"[1] {title_a} vs [2] {title_b}: ('q' to quit)").strip().lower()

            if user_choice == 'q' or user_choice == 'quit':
                break
            if user_choice not in ['1', '2']: 
                continue

            # Update Matches Played
            elo_df.loc[[index_a, index_b], 'matches_played'] += 1
            current_min_matches = elo_df.loc[bucket_indices, 'matches_played'].min()
            if current_min_matches > previous_min_matches:
                print(f"    All books have played at least {current_min_matches} match(es)")
                previous_min_matches = current_min_matches

            rating_a = elo_df.at[index_a, 'elo_score']
            rating_b = elo_df.at[index_b, 'elo_score']
            actual_result = 1 if user_choice == '1' else 0
            new_rating_a, new_rating_b = update_elo(rating_a, rating_b, actual_result)
            
            elo_df.at[index_a, 'elo_score'] = new_rating_a
            elo_df.at[index_b, 'elo_score'] = new_rating_b
            
        except KeyboardInterrupt:
            break
            
    return elo_df

def compute_continuous(elo_df):
    results = pd.Series(np.nan, index=elo_df.index, dtype=float)
    for stars in elo_df['original_rating'].dropna().unique():
        mask = elo_df['original_rating'] == stars
        subset = elo_df.loc[mask, 'elo_score']

        if len(subset) > 1 and subset.max() > subset.min():
            norm = (subset - subset.min()) / (subset.max() - subset.min())
            results.loc[mask] = (stars + (norm * 0.99) - 0.5).to_numpy(copy=True)
        else:
            results.loc[mask] = float(stars)
    results.index = elo_df['book_id']
    return results

def refine_ratings(target_df, rating_col, title_col='title', elo_path='data/elo_ratings.csv'):
    target_clean = target_df.dropna(subset=[rating_col]).copy()
    if os.path.exists(elo_path):
        elo_df = pd.read_csv(elo_path)
    else:
        elo_df = pd.DataFrame(columns=['book_id', 'original_rating', 'elo_score', 'matches_played'])
    elo_df = elo_df.set_index('book_id')
    
    common_books = elo_df.index.intersection(target_clean['book_id'])
    elo_df.loc[common_books, 'original_rating'] = target_clean.set_index('book_id').loc[common_books, rating_col]

    # Add new books
    new_books = target_clean[~target_clean['book_id'].isin(elo_df.index)]
    new_entries = pd.DataFrame({
        'original_rating': new_books[rating_col].values,
        'elo_score': 1200.0,
        'matches_played': 0
    }, index=new_books['book_id'])
    elo_df = pd.concat([elo_df, new_entries]).reset_index().rename(columns={'index': 'book_id'})

    # 3. Interactive Ranking
    if input(f"Rank '{rating_col}' buckets? (y/n): ").lower() == 'y':
        titles = dict(zip(target_df['book_id'], target_df[title_col]))
        for star in sorted(target_clean[rating_col].unique(), reverse=True):
            elo_df = run_interactive_ranking(elo_df, titles, star)
        elo_df.to_csv(elo_path, index=False)

    return compute_continuous(elo_df)

books_df = pd.read_csv("data/books.csv")
books_df['description'] = books_df['description'].fillna('').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
books_df['lang'] = books_df['lang'].apply(lambda x: "|".join(i.strip() for i in x.split(";")) if isinstance(x, str) else x)

gr_export = pd.read_csv('data/goodreads_library_export.csv')
gr_export['my_rating'] = gr_export['my_rating'].astype('UInt8').replace(0, np.nan)

my_refined = refine_ratings(gr_export, 'my_rating')
books_df = books_df.merge(gr_export[['book_id', 'my_rating']], left_on='book_id', right_on='book_id', how='left')
books_df = books_df.merge(my_refined.rename('my_refined'), left_on='book_id', right_index=True, how='left')

In [ ]:
# Prep embedding strings
def format_string_for_embedding(items, kind=None, truncate=0):
    if not isinstance(items, (list)) or len(items) == 0:
        return ""

    n = len(items)
    if n == 1:
        res = items[0]
    elif n > truncate > 1:
        res = f"{', '.join(items[:truncate])}, and {items[truncate]}"
    else:
        res = f"{', '.join(items[:-1])}{',' if n > 2 else ''} and {items[-1]}"
    
    prefix = f"{kind.capitalize()}{'s' if n > 1 else ''}: " if kind else ""
    return f"{prefix}{res}"

books_df['authors_post'] = books_df['authors'].str.split('|')
books_df['authors_post'] = books_df['authors_post'].apply(lambda x:format_string_for_embedding(x, truncate=4))

books_df['genres_post'] = books_df['genres'].str.split('|')
books_df['genres_post'] = books_df['genres_post'].apply(lambda x:format_string_for_embedding(x, kind='genre'))

books_df['desc_post'] = [[desc] if isinstance(desc, str) else [] for desc in books_df['description']]
books_df['desc_post'] = books_df['desc_post'].apply(lambda x:format_string_for_embedding(x, kind='description'))

def join_embedding_parts(title, authors, genres, desc):
    text = f"Book: {title}\n"
    if authors:
        text += f"Written by: {authors}\n"
    if genres:
        text += f"{genres}\n"
    if desc:
        text += f"{desc}" 
    return text

books_df['embedding_input'] = [
    join_embedding_parts(t, a, g, d) 
    for t, a, g, d in zip(books_df['title'], books_df['authors_post'], books_df['genres_post'], books_df['desc_post'])
]
id_to_string = books_df.set_index('book_id')['embedding_input']

In [ ]:
# Embed sentences
import os
import ollama
from tqdm import tqdm
import torch
from sklearn.preprocessing import Normalizer

PARAMS = 8
OLLAMA_MODEL = f"qwen3-embedding:{PARAMS}b"
embeddings_path = f'data/{PARAMS}b_embeddings.csv'
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
if os.path.exists(embeddings_path):
    embeddings = pd.read_csv(embeddings_path, index_col='book_id')
    embedding_cols = embeddings.columns.astype(int)
    loaded_book_ids = embeddings.index.values
    embeddings = torch.tensor(embeddings.values, dtype=torch.float32, device=device)
else:
    loaded_book_ids = np.array([])
    embeddings = None

current_book_ids = books_df['book_id'].values
missing_mask = ~np.isin(current_book_ids, loaded_book_ids)
missing_ids = current_book_ids[missing_mask]
if len(missing_ids) > 0:
    missing_strings = id_to_string.loc[missing_ids].tolist()
    batch_size = 128
    new_embeddings = []
    for i in tqdm(range(0, len(missing_strings), batch_size)):
        batch = missing_strings[i : i + batch_size]
        response = ollama.embed(model=OLLAMA_MODEL, input=batch)
        batch_embeddings = torch.tensor(response['embeddings'], dtype=torch.float32, device=device)
        new_embeddings.append(batch_embeddings)

    new_embeddings = torch.cat(new_embeddings, dim=0)
    if embeddings is not None:
        dim = max(embeddings.shape[1], new_embeddings.shape[1])
        if embeddings.shape[1] < dim:
            pad = torch.zeros((embeddings.shape[0], dim - embeddings.shape[1]), device=device)
            embeddings = torch.cat([embeddings, pad], dim=1)
        
        if new_embeddings.shape[1] < dim:
            pad = torch.zeros((new_embeddings.shape[0], dim - new_embeddings.shape[1]), device=device)
            new_embeddings = torch.cat([new_embeddings, pad], dim=1)

        full_embeddings = torch.empty((len(current_book_ids), dim), dtype=torch.float32, device=device)
        existing_idx_map = {book_id: pos for pos, book_id in enumerate(loaded_book_ids)}
        for i, book_id in enumerate(current_book_ids):
            if book_id in existing_idx_map:
                old_pos = existing_idx_map[book_id]
                full_embeddings[i] = embeddings[old_pos]
        
        missing_positions = np.where(missing_mask)[0]
        full_embeddings[missing_positions] = new_embeddings
    else:
        full_embeddings = new_embeddings

    save_df = pd.DataFrame(full_embeddings.cpu().numpy(), index=current_book_ids)
    save_df.index.name = 'book_id'
    save_df.to_csv(embeddings_path)

    embeddings = full_embeddings
    del full_embeddings, new_embeddings, save_df, batch_embeddings

In [ ]:
# Expand training set with friends' libraries
from sklearn.neighbors import KNeighborsRegressor
from scipy.stats import spearmanr

def safe_spearman(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 3 or len(y) < 3 or np.std(x) < 1e-9 or np.std(y) < 1e-9:
        return np.nan
    rho, _ = spearmanr(x, y)
    return float(rho) if not np.isnan(rho) else np.nan

def calibrate_friend_ratings(overlap_df, clip_min, clip_max, shrink_after=10):
    x = overlap_df['rating'].astype(float).to_numpy(copy=True)
    y = overlap_df['my_refined'].astype(float).to_numpy(copy=True)

    if len(overlap_df) >= 3 and np.std(x) > 1e-9:
        slope, intercept = np.polyfit(x, y, 1)
    else:
        slope = 1.0
        intercept = float(y.mean() - x.mean())

    # Small overlaps should only nudge the friend's scale toward yours.
    shrink = min(1.0, len(overlap_df) / shrink_after)
    slope = 1.0 + shrink * (slope - 1.0)
    intercept = shrink * intercept

    return slope, intercept

def get_similar_friend_ratings(
    books_df,
    friends_path,
    embeddings,
    min_overlap=5,
    min_correlation=0.3,
    min_similar_friends=2,
):
    friends = pd.read_csv(friends_path)
    friends['rating'] = friends['rating'].astype('UInt8').replace(0, np.nan)
    friends['list_id'] = friends['list_id'].astype(str)

    current_book_ids = books_df['book_id'].to_numpy()
    current_book_id_set = set(current_book_ids.tolist())
    friends = friends[friends['book_id'].isin(current_book_id_set)].dropna(subset=['rating']).copy()
    friends = friends.groupby(['list_id', 'book_id'], as_index=False)['rating'].mean()

    my_books = books_df[['book_id', 'my_refined']].dropna().copy()
    my_book_id_set = set(my_books['book_id'].tolist())
    my_lookup = my_books.set_index('book_id')['my_refined']
    clip_min = float(my_lookup.min())
    clip_max = float(my_lookup.max())

    if torch.is_tensor(embeddings):
        embedding_matrix = embeddings.detach().cpu().numpy()
    else:
        embedding_matrix = np.asarray(embeddings)
    book_id_to_idx = {book_id: i for i, book_id in enumerate(current_book_ids)}

    # Train KNN_me once — predicts my rating from book embeddings
    my_train_ids = my_books['book_id'].to_numpy()
    X_me = embedding_matrix[[book_id_to_idx[bid] for bid in my_train_ids]]
    y_me = my_lookup.loc[my_train_ids].to_numpy(copy=True)
    knn_me_k = min(max(2, int(np.sqrt(len(my_books)))), len(my_books))
    knn_me = KNeighborsRegressor(
        n_neighbors=knn_me_k, metric='cosine', weights='distance', n_jobs=-1
    )
    knn_me.fit(X_me, y_me)

    friend_scores = []
    calibrated_friend_rows = []

    for list_id, friend_df in friends.groupby('list_id'):
        overlap = friend_df.merge(my_books, on='book_id', how='inner') \
                           .dropna(subset=['rating', 'my_refined']).copy()
        overlap_count = len(overlap)
        if overlap_count < min_overlap:
            continue

        # Calibrate friend's ratings onto my scale
        slope, intercept = calibrate_friend_ratings(overlap, clip_min, clip_max)

        friend_df = friend_df.copy()
        friend_df['calibrated_rating'] = np.clip(
            slope * friend_df['rating'].astype(float).to_numpy(copy=True) + intercept,
            clip_min, clip_max,
        )
        calibrated_friend_rows.append(friend_df)

        # Partition into overlap / my-only / friend-only
        overlap_ids = set(overlap['book_id'].tolist())
        friend_book_id_set = set(friend_df['book_id'].tolist())
        my_only_ids = sorted(my_book_id_set - overlap_ids)
        friend_only_ids = sorted(friend_book_id_set - overlap_ids)

        # --- Overlap: real ratings on both sides ---
        calibrated_overlap = np.clip(
            slope * overlap['rating'].astype(float).to_numpy(copy=True) + intercept,
            clip_min, clip_max,
        )
        my_union = [overlap['my_refined'].to_numpy(copy=True)]
        friend_union = [calibrated_overlap]

        # Raw Spearman on overlap only (for diagnostics)
        raw_corr = safe_spearman(my_union[0], friend_union[0])

        # --- My-only books: real my rating, predict friend's calibrated rating ---
        if my_only_ids and len(friend_df) >= 2:
            friend_indexed = friend_df.set_index('book_id')
            X_friend = embedding_matrix[[book_id_to_idx[bid] for bid in friend_df['book_id']]]
            y_friend = friend_indexed.loc[friend_df['book_id'], 'calibrated_rating'].to_numpy(copy=True)

            knn_friend_k = min(max(2, int(np.sqrt(len(friend_df)))), len(friend_df))
            knn_friend = KNeighborsRegressor(
                n_neighbors=knn_friend_k, metric='cosine', weights='distance', n_jobs=-1
            )
            knn_friend.fit(X_friend, y_friend)

            X_my_only = embedding_matrix[[book_id_to_idx[bid] for bid in my_only_ids]]
            predicted_friend = knn_friend.predict(X_my_only)

            my_union.append(my_lookup.loc[my_only_ids].to_numpy(copy=True))
            friend_union.append(predicted_friend)

        # --- Friend-only books: predict my rating, real friend calibrated rating ---
        if friend_only_ids:
            X_friend_only = embedding_matrix[[book_id_to_idx[bid] for bid in friend_only_ids]]
            predicted_me = knn_me.predict(X_friend_only)

            friend_indexed = friend_df.set_index('book_id')
            real_friend = friend_indexed.loc[friend_only_ids, 'calibrated_rating'].to_numpy(copy=True)

            my_union.append(predicted_me)
            friend_union.append(real_friend)

        # Synthetic Spearman over full union
        my_union = np.concatenate(my_union)
        friend_union = np.concatenate(friend_union)
        synthetic_corr = safe_spearman(my_union, friend_union)

        # Blend raw and synthetic correlations 50/50 to balance direct evidence vs taste extension
        if not np.isnan(raw_corr) and not np.isnan(synthetic_corr):
            score = 0.5 * raw_corr + 0.5 * synthetic_corr
        elif not np.isnan(raw_corr):
            score = raw_corr
        else:
            score = synthetic_corr

        if np.isnan(score):
            continue

        friend_scores.append({
            'list_id': list_id,
            'overlap_count': overlap_count,
            'union_count': len(my_union),
            'friend_books': len(friend_df),
            'slope': slope,
            'intercept': intercept,
            'raw_corr': raw_corr,
            'synthetic_corr': synthetic_corr,
            'score': score,
        })

    friend_scores = pd.DataFrame(friend_scores).sort_values('score', ascending=False).reset_index(drop=True)
    if friend_scores.empty:
        return pd.Series(dtype=float), [], friend_scores

    # Select friends above correlation threshold
    selected = friend_scores[friend_scores['score'] >= min_correlation].copy()
    minimum_selected = min(min_similar_friends, len(friend_scores))
    if len(selected) < minimum_selected:
        selected = friend_scores.head(minimum_selected).copy()

    similar_friends = selected['list_id'].tolist()
    similar_friend_ratings = pd.concat(calibrated_friend_rows, ignore_index=True)
    similar_friend_ratings = similar_friend_ratings.merge(
        selected[['list_id', 'score']].rename(columns={'score': 'friend_weight'}),
        on='list_id', how='inner',
    )
    similar_friend_ratings = similar_friend_ratings[
        ~similar_friend_ratings['book_id'].isin(my_book_id_set)
    ].copy()
    if similar_friend_ratings.empty:
        return pd.Series(dtype=float), similar_friends, friend_scores

    similar_friend_ratings['weighted_rating'] = (
        similar_friend_ratings['calibrated_rating'] * similar_friend_ratings['friend_weight']
    )
    similar_friend_ratings = similar_friend_ratings.groupby('book_id').agg(
        weighted_rating=('weighted_rating', 'sum'),
        total_weight=('friend_weight', 'sum'),
        supporting_friends=('list_id', 'nunique'),
    )
    similar_friend_ratings['rating'] = (
        similar_friend_ratings['weighted_rating'] / similar_friend_ratings['total_weight']
    )

    return similar_friend_ratings['rating'], similar_friends, friend_scores


similar_friend_ratings, similar_friends, friend_scores = get_similar_friend_ratings(
    books_df,
    "data/friend_ratings.csv",
    embeddings,
    min_correlation=0.3,
)
books_df['training_ratings'] = books_df['my_refined'].fillna(books_df['book_id'].map(similar_friend_ratings))

my_rating_count = len(books_df[books_df['my_rating'].notna()])
total_training_count = len(books_df[books_df['training_ratings'].notna()])
print(f"My ratings({my_rating_count}) + Others({total_training_count - my_rating_count})")


In [ ]:
similar_friends

In [ ]:
# # MRL dim reduction
train_size = (~books_df['training_ratings'].isna()).sum()
mrl_dimensions = 32 # Min recommended MRL dimensions according to qwen3-embedding github
while mrl_dimensions*2 < train_size:
    mrl_dimensions*=2

# embeddings = embeddings[:, :mrl_dimensions*2] # qwen-3 embedding supports MRL
# norm = Normalizer(norm='l2')
# embeddings = norm.fit_transform(embeddings)


# embeddings = embeddings.loc[books_df['book_id']].values

# embeddings = embeddings[:, :mrl_dimensions*2] # qwen-3 embedding supports MRL
# norm = Normalizer(norm='l2')
# embeddings = norm.fit_transform(embeddings)

# embeddings = torch.tensor(embeddings, dtype=torch.float32)

In [ ]:
# Build adjacency matrix
from torch_geometric.utils import add_self_loops
from torch_geometric.nn.conv.gcn_conv import gcn_norm

id_to_idx = {id: i for i, id in enumerate(books_df['book_id'])}

edge_indices = []
for idx, row in tqdm(books_df.iterrows(), total=len(books_df)):
    current_idx = id_to_idx[row['book_id']]
    if pd.isna(row['similar_books']):
        continue
    for item in row['similar_books'].split('|'):
        try:
            target_id = int(item.split(':')[0])
            if target_id in books_df['book_id'].values:
                target_idx = id_to_idx[target_id]
                edge_indices.append([current_idx, target_idx])
                edge_indices.append([target_idx, current_idx])
        except (ValueError, IndexError):
            continue

if not edge_indices:
    edge_index = torch.tensor([[], []], dtype=torch.long)
else:
    edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()

edge_index_with_loops, _ = add_self_loops(edge_index, num_nodes=embeddings.size(0))
edge_index_norm, edge_weight_norm = gcn_norm(edge_index_with_loops, num_nodes=embeddings.size(0))

edge_index_norm = edge_index_norm
edge_weight_norm = edge_weight_norm

adj_matrix = torch.sparse_coo_tensor(edge_index_norm, edge_weight_norm, (embeddings.size(0), embeddings.size(0)))

In [ ]:
import nevergrad as ng
from sklearn.preprocessing import Normalizer, MinMaxScaler
from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import BayesianRidge
from sklearn.svm import SVR
from sklearn.metrics import ndcg_score, mean_squared_error
from scipy.stats import spearmanr
from matplotlib import pyplot as plt


def prep_optimization(training_col='training_ratings', budget=300):

    def objective(num_propagations, 
                knn_neighbors,
                brr_alpha_1, brr_alpha_2, brr_lambda_1, brr_lambda_2, brr_uncertainty_penalty,
                svr_C, svr_epsilon,
                knn_weight, brr_weight
                ):
        
        all_embeddings = precomputed_embeddings[num_propagations]
        X = all_embeddings[training_mask]
        y = my_ratings_scaled

        y_reals = []
        y_preds = []
        skf = KFold(n_splits=10, shuffle=True, random_state=42)
        for train_idx, test_idx in skf.split(X):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            knn = KNeighborsRegressor(
                n_neighbors=knn_neighbors,
                metric='cosine', 
                weights='distance',
                n_jobs=-1,
                )
            knn.fit(X_train, y_train)
            knn_pred = knn.predict(X_test)

            brr = BayesianRidge(
                alpha_1=brr_alpha_1,
                alpha_2=brr_alpha_2,
                lambda_1=brr_lambda_1,
                lambda_2=brr_lambda_2,
                compute_score=True
                )
            brr.fit(X_train, y_train)
            brr_mu, brr_std = brr.predict(X_test, return_std=True)
            brr_pred = brr_mu - (brr_uncertainty_penalty * brr_std)

            svr = SVR(
                kernel='rbf',
                gamma='scale',
                C=svr_C, 
                epsilon=svr_epsilon
                ) 
            svr.fit(X_train, y_train)
            svr_pred = svr.predict(X_test)

            remaining_weight = 1 - knn_weight
            actual_brr_weight = brr_weight * remaining_weight
            svr_weight = remaining_weight - actual_brr_weight
            final_pred = (knn_weight * knn_pred) + (actual_brr_weight * brr_pred) + (svr_weight * svr_pred)

            y_reals.append(y_test)
            y_preds.append(final_pred)

        y_reals = np.concatenate(y_reals)
        y_preds = np.concatenate(y_preds)
        
        mse = mean_squared_error(y_reals, y_preds)
        ndcg = ndcg_score([y_reals], [y_preds])
        if np.std(y_preds) < 1e-9:
            spearman = 0
        else:
            spearman, _ = spearmanr(y_reals, y_preds)
            if np.isnan(spearman):
                spearman = 0

        return mse + (1 - ndcg) + (1 - spearman)


    training_mask = ~books_df[training_col].isna()
    my_ratings = books_df.loc[training_mask, training_col].values
    scaler = MinMaxScaler(feature_range=(0, 1))
    my_ratings_scaled = scaler.fit_transform(my_ratings.reshape(-1, 1)).flatten()

    parametrization = ng.p.Instrumentation(
        num_propagations = ng.p.Scalar(lower=0, upper=max_propagations).set_integer_casting(),

        knn_neighbors = ng.p.Scalar(lower=1, upper=mrl_dimensions//3).set_integer_casting(),
        brr_alpha_1=ng.p.Scalar(lower=1e-7, upper=1e-3),
        brr_alpha_2=ng.p.Scalar(lower=1e-7, upper=1e-3),
        brr_lambda_1=ng.p.Log(lower=1e-6, upper=1e-1),
        brr_lambda_2=ng.p.Log(lower=1e-6, upper=1e-1),
        brr_uncertainty_penalty=ng.p.Scalar(lower=0, upper=2.0),
        svr_C = ng.p.Log(lower=1e-3, upper=1e2),
        svr_epsilon = ng.p.Scalar(lower=0.0, upper=1.0),

        knn_weight=ng.p.Scalar(lower=0, upper=1), 
        brr_weight=ng.p.Scalar(lower=0, upper=1), 
    )

    optimizer = ng.optimizers.NGOpt(parametrization=parametrization, budget=budget)
    best_loss = float('inf')
    y = []
    with tqdm(total=budget) as pbar:
        for i in range(budget):
            x = optimizer.ask()
            loss = objective(*x.args, **x.kwargs)
            optimizer.tell(x, loss)
            y.append(loss)
            if loss < best_loss:
                best_loss = loss
            pbar.update(1)
            if i % 10 == 0:
                pbar.set_description(f"Best Loss: {best_loss:.4f}")

    best_params = optimizer.provide_recommendation().kwargs

    plt.plot(range(budget), sorted(y, reverse=True))
    plt.show()
    plt.plot(range(budget), y)
    plt.show()

    return best_params

propagated = embeddings.clone().cpu()
norm_l2 = Normalizer(norm='l2')
precomputed_embeddings = [norm_l2.transform(propagated.numpy())]

max_propagations = 2
for _ in range(max_propagations):
    propagated = torch.sparse.mm(adj_matrix, propagated)
    precomputed_embeddings.append(norm_l2.transform(propagated.numpy()))
del propagated, adj_matrix

if input("run optimization? (y/n)").lower() == 'y':
    friend_params = prep_optimization(training_col='training_ratings', budget=200)
    solo_params = prep_optimization(training_col='my_refined', budget=200)
else:
    # Hardcoded params from previous runs - to save time
    friend_params = {
        'num_propagations': 0,
        'knn_neighbors': 18,
        'brr_alpha_1': 0.0007083523294476891,
        'brr_alpha_2': 0.0005052291699790206,
        'brr_lambda_1': 6.723461212511323e-06,
        'brr_lambda_2': 0.00065032051532833,
        'brr_uncertainty_penalty': 1.124317064784675,
        'svr_C': 0.045512095772648135,
        'svr_epsilon': 0.45635064943633097,
        'knn_weight': 0.6072006217417109,
        'brr_weight': 0.7469140779723126
        }
    solo_params = {
        'num_propagations': 1,
        'knn_neighbors': 39,
        'brr_alpha_1': 0.0007917501343982147,
        'brr_alpha_2': 0.0003673094481243756,
        'brr_lambda_1': 0.0009261898325379562,
        'brr_lambda_2': 0.0004845024834776367,
        'brr_uncertainty_penalty': 1.4875061034371395,
        'svr_C': 2.0283092484268437,
        'svr_epsilon': 0.2673464313322004,
        'knn_weight': 0.11589067396260068,
        'brr_weight': 0.1852941582795484
        }

In [ ]:
# Run optimized hyperparams
def run_optimized(best_params, training_col='training_ratings'):
    all_embeddings = precomputed_embeddings[best_params['num_propagations']]

    training_mask = ~books_df[training_col].isna()
    my_ratings = books_df.loc[training_mask, training_col].values
    scaler = MinMaxScaler(feature_range=(0, 1))
    my_ratings_scaled = scaler.fit_transform(my_ratings.reshape(-1, 1)).flatten()

    X_train = all_embeddings[training_mask]
    y_train = my_ratings_scaled

    knn = KNeighborsRegressor(
        n_neighbors=best_params['knn_neighbors'],
        metric='cosine', 
        weights='distance',
        n_jobs=-1,
        )
    knn.fit(X_train, y_train)
    knn_pred = knn.predict(all_embeddings)

    brr = BayesianRidge(
        alpha_1=best_params['brr_alpha_1'],
        alpha_2=best_params['brr_alpha_2'],
        lambda_1=best_params['brr_lambda_1'],
        lambda_2=best_params['brr_lambda_2'],
        compute_score=True
        )
    brr.fit(X_train, y_train)
    brr_mu, brr_std = brr.predict(all_embeddings, return_std=True)
    brr_pred = brr_mu - (best_params['brr_uncertainty_penalty'] * brr_std)

    svr = SVR(
        kernel='rbf',
        gamma='scale',
        C=best_params['svr_C'], 
        epsilon=best_params['svr_epsilon']
        ) 
    svr.fit(X_train, y_train)
    svr_pred = svr.predict(all_embeddings)

    knn_weight = best_params['knn_weight']
    brr_weight = best_params['brr_weight']

    remaining_weight = 1 - knn_weight
    brr_weight *= remaining_weight
    svr_weight = remaining_weight - brr_weight

    knn_pred = scaler.inverse_transform(knn_pred.reshape(-1, 1)).flatten()
    brr_pred = scaler.inverse_transform(brr_pred.reshape(-1, 1)).flatten()
    svr_pred = scaler.inverse_transform(svr_pred.reshape(-1, 1)).flatten()
    final_pred = (knn_weight * knn_pred) + (brr_weight * brr_pred) + (svr_weight * svr_pred)

    # return knn_pred, brr_pred, svr_pred, final_pred
    return final_pred

# combined_knn_pred, combined_brr_pred, combined_svr_pred, combined_final_pred = run_optimized(combined_params, training_col='training_ratings')
# solo_knn_pred, solo_brr_pred, solo_svr_pred, solo_final_pred = run_optimized(solo_params, training_col='my_refined')
friend_final_pred = run_optimized(friend_params, training_col='training_ratings')
solo_final_pred = run_optimized(solo_params, training_col='my_refined')

In [ ]:
# Final combined ratings
from sklearn.preprocessing import MinMaxScaler

star_cols = [col for col in books_df.columns if col.endswith('star')]
books_df['rating_count'] = books_df[star_cols].sum(axis=1)

C = books_df['avg_rating'].mean()
m = books_df['rating_count'].quantile(0.10) 

def weighted_rating(x, m=m, C=C):
    v = float(x['rating_count'])
    R = float(x['avg_rating'])
    if v == 0: 
        return C
    return (v / (v + m) * R) + (m / (v + m) * C)
count_adjusted = books_df.apply(weighted_rating, axis=1)

scaler = MinMaxScaler()
solo_pred = scaler.fit_transform(solo_final_pred.reshape(-1, 1))
friend_pred = scaler.fit_transform(friend_final_pred.reshape(-1, 1))
count_adjusted = scaler.fit_transform(count_adjusted.values.reshape(-1, 1))

books_df['friend_pred_rating'] = friend_pred
books_df['solo_pred_rating'] = solo_pred
books_df['count_adjusted_rating'] = count_adjusted

books_df['pred_rating'] = friend_pred * solo_pred
books_df['final_rating'] = count_adjusted * friend_pred * solo_pred #/ (np.log1p(books_df['num_pages']))
del solo_pred, solo_final_pred, friend_pred, friend_final_pred, count_adjusted, scaler, precomputed_embeddings

In [ ]:
# from sklearn.decomposition import PCA
# import plotly.express as px

genre_matrix = books_df['genres'].str.get_dummies(sep='|')
# adjacency_matrix = genre_matrix.T.dot(genre_matrix) # The dot product of the transposed matrix with itself yields the co-occurrence matrix
# genre_counts = np.diag(adjacency_matrix) # The diagonal contains the exact count of each genre
# pca = PCA(n_components=3)
# genre_3d_coords = pca.fit_transform(adjacency_matrix)

# plot_df = pd.DataFrame({
#     'Genre': adjacency_matrix.index,
#     'Count': genre_counts,
#     'X': genre_3d_coords[:, 0],
#     'Y': genre_3d_coords[:, 1],
#     'Z': genre_3d_coords[:, 2]
# })

# fig = px.scatter_3d(
#     plot_df, 
#     x='X', y='Y', z='Z',
#     text='Genre',           # Show genre names next to dots
#     size='Count',           # Larger spheres for more frequent genres
#     color='Count',          # Color gradient based on frequency
#     title='3D Genre Co-occurrence Space'
# )

# # Improve layout for readability
# fig.update_traces(textposition='top center', marker=dict(sizemin=3))
# fig.update_layout(scene=dict(xaxis_title='PCA 1', yaxis_title='PCA 2', zaxis_title='PCA 3'))
# fig.show()
display(genre_matrix.sum().sort_values(ascending=False))
del genre_matrix

In [ ]:
# books_df[books_df['my_rating'].isna()][display_cols].sort_values(by='final_rating', ascending=False)

display_cols = ['book_id', 'title', 'authors', 'year', 'genres', 'num_pages', 'solo_pred_rating', 'friend_pred_rating', 'count_adjusted_rating', 'pred_rating', 'final_rating']
rating_cols = [col for col in display_cols if col.endswith('rating')]

not_read = books_df[books_df['my_rating'].isna()]
not_read = pd.concat([not_read.nlargest(500, col) for col in rating_cols]).drop_duplicates()[display_cols]

not_read.to_csv('data/recommendations.csv', index=False)

In [ ]:
not_read = pd.read_csv('data/recommendations.csv')
not_read#[not_read['genres'].str.contains(r'\bNonfiction\b', na=False)]